# 第 14 课 · 用 SFT 预热诊断奖励冷启动

第 12 课有 3 步出现非零梯度，但固定测试的合法输出率与准确率均为零。可能的问题包括输出协议未学会、输入含 UNK、奖励过于稀疏，以及训练预算不足。本课先验证一个可检验的假设：监督预热能否让模型稳定输出合法答案？

我们显式改变协议：补上词表中的 `or`，使用三位数，固定新的 240/80/80 训练、验证、测试划分，各组奇偶平衡。三位数未用于第 12 课，但任务的个位规则相同。这是新实验，不把新旧分数差直接解释为单一改动的因果效果。

需要 CUDA、transformers、tokenizers、pandas、pyarrow。无需下载模型。先关闭其他训练内核；每次完整执行创建独立目录。可在 [配套源码](support/advanced14/experiment.py) 查看所有实现。

历史验证记录：参考环境、种子 14014、固定 200 步 SFT 后，80 道测试题的合法率与准确率从冷启动的 0% 提高到 100%。再追加真实 verl 的 10 步 GRPO，准确率仍为 100%，其中 1 步有非零梯度。完整记录见 [验证说明](support/docs/VALIDATION.md)；这是单次实测，不是预设的运行断言。

## 1 · 检查协议与划分

在训练前核对：提示没有 UNK；标准答案不出现在提示尾部；train/validation/test 数值不交叉。问题中的 yes/no 是候选协议，不是正确标签。三位数测试仍是同分布组合评估，不是通用数学能力。


In [ ]:
import gc, importlib.util, json, os, time
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
spec = importlib.util.spec_from_file_location('lesson14', Path('support/advanced14/experiment.py'))
lesson = importlib.util.module_from_spec(spec); spec.loader.exec_module(lesson)
assert torch.cuda.is_available(), '请选择支持 CUDA 的内核'
SEED = 14014
tokenizer, initial, splits = lesson.prepare(SEED)
initial = initial.cuda()
assert not(set(splits['train']) & set(splits['test']))
assert not(set(splits['validation']) & (set(splits['train']) | set(splits['test'])))
for numbers in splits.values(): lesson.batch(tokenizer, numbers, 'cuda')
print({k: len(v) for k, v in splits.items()})
example = splits['train'][0]
print('提示:', lesson.prompt(example))
print('token:', tokenizer.convert_ids_to_tokens(lesson.encode_prompt(tokenizer, example)))
print('参数量:', sum(p.numel() for p in initial.parameters()))
base = Path(os.environ.get('VERL_ADV14_BASE', str(Path.home()/'verl-support/advanced14/runs')))
run = base/f'{time.strftime("%Y%m%d-%H%M%S")}-{time.time_ns()}'
run.mkdir(parents=True)
print('本次目录:', run)


## 2 · 固定预算预热，只监测验证集

从同一随机起点保留冻结基线，另一个副本做 200 步 SFT。每步 32 题、学习率 0.001，只计算答案与 EOS 的损失，每 20 步展示验证结果。固定最后一步用于评估，不按最终测试挑模型。

合法输出率衡量格式，准确率衡量任务，EOS 率衡量停止行为。三者要分开看。监督预热同时学习了格式和任务，不能据此归因于“只修复格式”。


In [ ]:
def live(history):
    clear_output(wait=True)
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(frame.step, frame.loss); axes[0].set(title='SFT loss', xlabel='Step')
    axes[1].plot(frame.step, frame.valid_rate, label='Valid')
    axes[1].plot(frame.step, frame.accuracy, label='Accuracy')
    axes[1].set(ylim=(-.05, 1.05), title='Validation only', xlabel='Step'); axes[1].legend()
    plt.tight_layout(); plt.show(); plt.close(fig)
    print(frame.tail(1).to_string(index=False))
model, history = lesson.warmup(tokenizer, initial, splits, steps=200, seed=SEED, callback=live)
manifest = lesson.export(run, tokenizer, model, initial, splits, history, seed=SEED)
print('模型、数据与配置已导出:', run)


## 3 · 预先固定的测试与完整记录

现在才对测试集运行冻结基线和 SFT 模型。多数类别基线为 50%，由预先平衡的数据设计决定。固定猜 yes 也能得到 100% 合法率，因此合法率改善本身不足以证明任务学会。

如果 SFT 已达到很高准确率，后续 GRPO 可能只有很少混合奖励组。此时应研究有效组比例与资源成本，不应为了制造提升而事后挑一个较差起点。若要比较预热步数，另做验证集消融并预留新的最终测试。


In [ ]:
rows = []
for stage, obj in [('cold', initial), ('sft', model)]:
    rows += lesson.evaluate(obj, tokenizer, splits['test'], stage, 'test')
predictions = pd.DataFrame(rows)
summary = predictions.groupby('stage')[['valid','correct','eos']].mean()
display(summary)
display(predictions[predictions.stage=='sft'].head(10))
out = Path('support/experiment14_output'); out.mkdir(exist_ok=True)
predictions.to_csv(out/'predictions.csv', index=False)
summary.to_csv(out/'summary.csv')
pd.DataFrame(history).to_csv(out/'history.csv', index=False)
(out/'manifest.json').write_text(json.dumps(manifest, indent=2))
(out/'latest_run.txt').write_text(str(run))
print('多数类别基线: 0.5；完整预测已保存。')


## 4 · 接回真实 verl：可选的完整训练

导出的 `model/`、Parquet 和奖励函数可以交给 verl。下格只打印命令并释放当前模型显存，训练需要在终端执行，避免 Notebook 持有的模型与 vLLM 竞争显存。脚本沿用第 12 课的单卡配置，固定 10 步，KL 关闭；本课不会自动启动训练。

这里的对照是 SFT 冻结模型与 SFT 后再 GRPO，尚未包含等预算继续 SFT，因此只能回答“追加 GRPO 后发生了什么”。吞吐、梯度及结果都以本轮日志为准。训练成功后才写 `latest_successful_grpo.txt`，失败目录保留用于诊断。


In [ ]:
import shlex
del model, initial
gc.collect(); torch.cuda.empty_cache()
print('在已安装 verl 的项目根目录执行：')
print('VERL_ADV14_ROOT=' + shlex.quote(str(run)) + ' bash tutorial/support/advanced14/run_grpo.sh')
print('完成后在下一格启用读取；未运行时不生成 GRPO 成绩。')


## 5 · 读取真实检查点，或明确标记尚未运行

仅加载自己生成的可信检查点。FSDP 文件可能包含 pickle 对象。默认跳过读取，避免误把旧目录当作新训练。启用后对同一测试题作配对比较，第 15 课介绍如何量化不确定性。


In [ ]:
READ_GRPO = False
if READ_GRPO:
    completed = Path((run/'latest_successful_grpo.txt').read_text().strip())
    display(lesson.evaluate_grpo(completed, out))
else:
    print('GRPO 尚未在本 Notebook 中评估；当前结果仅包含冷启动与 SFT。')


## 6 · 工程验收与下一次实验

提交运行 manifest、全部测试预测、学习曲线，以及一个能被证据否定的结论。例如：“固定 200 步 SFT 是否提高了协议合法率？”

进一步研究请预先注册三个分支：冻结 SFT、继续 SFT、SFT 后 GRPO。记录每个分支的 GPU 时间、生成 token 数、监督标签使用量与至少三个随机种子，不将更新次数相同称为算力相同。

下一项消融应把“补齐词表”“仅格式预热”“任务预热”分别控制，使用新的评估划分。每次只改变一个因素，保留失败运行。课后参考：[verl GRPO 示例](https://verl.readthedocs.io/en/latest/examples/grpo_example.html)。
